# AAV5 — DeepProfileMLP sur sel_org2, pleine échelle

`sel_org2` est, parmi toutes les cibles AAV5 testées cette session, celle qui donne
systématiquement le meilleur Pearson r held-out (Potts non filtré : +0.399, ProfileMLP shallow
sur le CSV trié : +0.438 — le meilleur résultat MLP obtenu jusqu'ici, cf.
`AAV5_SEL_profile_model_before_after_sorting.ipynb`) — et c'est aussi la cible avec le PLUS de
lignes mesurées après `viab` (486 844 sur le CSV brut, 381 627 sur le trié — largement plus que
`sel_org1`/`sel_org3`).

Ce notebook exploite cette masse de données : au lieu du cap `N_FIT=N_EVAL=150 000` des
notebooks précédents, on entraîne sur **tout le fit set disponible** (la moitié de chaque CSV,
pas de sous-échantillonnage), et on compare deux architectures sur exactement le même split :

- **`ShallowProfileMLP`** — l'archi standard du projet (Linear+BatchNorm+Dropout+GELU ×2, 140
  entrées, ~27k paramètres), rebaseline ici dans ce régime "pleine donnée" pour comparaison.
- **`DeepProfileMLP`** — reprise telle quelle de
  `Modelization_V1/notebooks/viability_parameter_sweeps/diversity_sweep_deeper_mlp.ipynb`
  (~90k paramètres, 3.3x plus gros) : embedding par position partagé (Linear 20→16 + GELU)
  moyenné sur les 7 positions (branche "profile"), branche pairwise (MLP sur les 21 paires de
  positions concaténées, avg pool) plus expressive qu'un simple terme bilinéaire, concaténées au
  one-hot brut, puis tête dense à 4 couches (256-128-64-32) au lieu de 2.

Sur les **2 CSV** déjà comparés pour `sel_org2` (`AAV5_organoides.csv` brut,
`AAV5_organoides_sorted.csv` trié) — 4 modèles au total. Mêmes métriques que les notebooks
`ProfileMLP` précédents : Pearson r held-out + recovery top-k% (`precision_at_k`,
`plot_topk_recovery`). Terminologie du projet respectée : log enrichment, jamais "score".

### 0. Setup

In [ ]:
import sys, os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.model_selection import train_test_split

_root = next(p for p in [Path.cwd(), *Path.cwd().parents] if p.name == "Modelization_V2")
sys.path.insert(0, str(_root / "lib"))
os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"

import jax
import jax.numpy as jnp
from flax import nnx
from typing import Optional
import optax
from tqdm.auto import tqdm

from analysisV1 import AA_LABELS, pearson, precision_at_k, plot_topk_recovery

print(f"JAX backend: {jax.default_backend()} -- devices: {jax.devices()}")

L, A = 7, 20

### 1. Chargement -- sel_org2, brut + trié

In [ ]:
SEL_ORG2_COL = "log2_enrichissement_organoide_2_adn_sur_virus"
use_cols = ["sequence", SEL_ORG2_COL]
dtypes = {"sequence": "string", SEL_ORG2_COL: "float32"}

datasets = {}
for tag, fname in [("brut", "AAV5_organoides.csv"), ("trié", "AAV5_organoides_sorted.csv")]:
    p = Path(fname)
    if not p.exists():
        p = _root / "notebooks/notebooks/Selectivity/AAV5" / fname
    assert p.exists(), p
    df = pd.read_csv(p, usecols=use_cols, dtype=dtypes)
    df["sequence"] = df["sequence"].astype("string")
    datasets[tag] = df
    n_finite = int(np.isfinite(df[SEL_ORG2_COL].to_numpy()).sum())
    print(f"{tag:5s} {fname:28s} {len(df):,} variants  (sel_org2 fini: {n_finite:,})")

lut = np.zeros(256, dtype=np.int64)
for i, aa in enumerate(AA_LABELS):
    lut[ord(aa)] = i

def encode(df):
    return lut[np.frombuffer("".join(df["sequence"]).encode("ascii"), np.uint8)].reshape(len(df), L)

seq_matrices = {tag: encode(df) for tag, df in datasets.items()}

### 2. Architectures -- Shallow (baseline) vs Deep

`DeepProfileMLP` reprise verbatim de `diversity_sweep_deeper_mlp.ipynb` (Modelization_V1) --
mêmes classes `PositionEmbed`/`PairwiseHead`/`DeepProfileMLP`, adaptées ici pour la boucle
d'entraînement `nnx.scan` déjà utilisée dans `AAV5_SEL_profile_model_before_after_sorting.ipynb`
(signature `model(x, train=..., rngs=...)` compatible entre les deux archis).

In [ ]:
class ShallowProfileMLP(nnx.Module):
    """Archi standard du projet (Linear+BatchNorm+Dropout+GELU x2, ~27k params)."""

    def __init__(self, input_dim: int, hidden_dims: tuple[int, int] = (128, 64),
                 dropout_rate: float = 0.1, *, rngs: nnx.Rngs):
        h1, h2 = hidden_dims
        self.linear1    = nnx.Linear(input_dim, h1, rngs=rngs)
        self.batchnorm1 = nnx.BatchNorm(h1, use_running_average=False, rngs=rngs)
        self.dropout1   = nnx.Dropout(rate=dropout_rate, rngs=rngs)
        self.linear2    = nnx.Linear(h1, h2, rngs=rngs)
        self.batchnorm2 = nnx.BatchNorm(h2, use_running_average=False, rngs=rngs)
        self.dropout2   = nnx.Dropout(rate=dropout_rate, rngs=rngs)
        self.linear3    = nnx.Linear(h2, 1, rngs=rngs)

    def __call__(self, x: jax.Array, *, train: bool, rngs: Optional[nnx.Rngs] = None) -> jax.Array:
        x = self.linear1(x)
        x = self.batchnorm1(x, use_running_average=not train)
        x = self.dropout1(x, deterministic=not train, rngs=rngs)
        x = nnx.gelu(x)
        x = self.linear2(x)
        x = self.batchnorm2(x, use_running_average=not train)
        x = self.dropout2(x, deterministic=not train, rngs=rngs)
        x = nnx.gelu(x)
        return self.linear3(x).squeeze(-1)


class PositionEmbed(nnx.Module):
    """Embedding par position PARTAGE : meme Linear+GELU appliquee aux 7 positions (Deep-Sets)."""
    def __init__(self, A: int, d_emb: int, *, rngs: nnx.Rngs):
        self.linear = nnx.Linear(A, d_emb, rngs=rngs)
    def __call__(self, x_pos):  # (batch, L, A)
        return nnx.gelu(self.linear(x_pos))  # (batch, L, d_emb)


class PairwiseHead(nnx.Module):
    """Pour chaque paire de positions (i,j) : concat des embeddings, MLP partage, avg-pool sur
    les 21 paires -- plus expressif qu'un terme bilineaire brut."""
    def __init__(self, L: int, d_emb: int, d_pair: int, *, rngs: nnx.Rngs):
        self.pairs_i = jnp.array([i for i in range(L) for j in range(i + 1, L)])
        self.pairs_j = jnp.array([j for i in range(L) for j in range(i + 1, L)])
        self.linear1 = nnx.Linear(2 * d_emb, 2 * d_pair, rngs=rngs)
        self.linear2 = nnx.Linear(2 * d_pair, d_pair, rngs=rngs)
    def __call__(self, emb):  # (batch, L, d_emb)
        e_i = emb[:, self.pairs_i, :]
        e_j = emb[:, self.pairs_j, :]
        pair_feat = jnp.concatenate([e_i, e_j], axis=-1)
        h = nnx.gelu(self.linear1(pair_feat))
        h = self.linear2(h)
        return jnp.mean(h, axis=1)  # avg pool over the 21 pairs


class DeepProfileMLP(nnx.Module):
    """Embedding par position -> [profile pooled, pairwise pooled, one-hot brut] concatenes ->
    tete dense a 4 couches (256-128-64-32), ~90k params (3.3x ShallowProfileMLP)."""

    def __init__(self, L: int, A: int, d_emb: int = 16, d_pair: int = 16,
                 hidden_dims: tuple = (256, 128, 64, 32), dropout_rate: float = 0.1,
                 *, rngs: nnx.Rngs):
        self.L, self.A = L, A
        self.pos_embed = PositionEmbed(A, d_emb, rngs=rngs)
        self.pairwise  = PairwiseHead(L, d_emb, d_pair, rngs=rngs)
        input_dim = d_emb + d_pair + L * A
        dims = (input_dim,) + hidden_dims
        n_layers = len(hidden_dims)
        self.linears    = nnx.List([nnx.Linear(dims[i], dims[i + 1], rngs=rngs) for i in range(n_layers)])
        self.batchnorms = nnx.List([nnx.BatchNorm(dims[i + 1], use_running_average=False, rngs=rngs) for i in range(n_layers)])
        self.dropouts   = nnx.List([nnx.Dropout(rate=dropout_rate, rngs=rngs) for _ in range(n_layers)])
        self.out        = nnx.Linear(hidden_dims[-1], 1, rngs=rngs)

    def __call__(self, x_flat: jax.Array, *, train: bool, rngs: Optional[nnx.Rngs] = None) -> jax.Array:
        batch = x_flat.shape[0]
        x_pos = x_flat.reshape(batch, self.L, self.A)
        emb             = self.pos_embed(x_pos)
        profile_pooled  = jnp.mean(emb, axis=1)
        pairwise_pooled = self.pairwise(emb)
        h = jnp.concatenate([profile_pooled, pairwise_pooled, x_flat], axis=-1)
        for linear, bn, drop in zip(self.linears, self.batchnorms, self.dropouts):
            h = linear(h)
            h = bn(h, use_running_average=not train)
            h = nnx.gelu(h)
            h = drop(h, deterministic=not train, rngs=rngs)
        return self.out(h).squeeze(-1)


_rngs = nnx.Rngs(0)
_n_shallow = sum(p.size for p in jax.tree.leaves(nnx.state(ShallowProfileMLP(input_dim=L * A, rngs=_rngs), nnx.Param)))
_n_deep    = sum(p.size for p in jax.tree.leaves(nnx.state(DeepProfileMLP(L=L, A=A, rngs=_rngs), nnx.Param)))
print(f"ShallowProfileMLP: {_n_shallow:,} params")
print(f"DeepProfileMLP:    {_n_deep:,} params  ({_n_deep / _n_shallow:.1f}x)")

In [ ]:
@nnx.jit
def train_step(model, optimizer, x, y, rngs):
    def loss_fn(model, rngs):
        y_pred = model(x, train=True, rngs=rngs)
        return jnp.mean((y_pred - y) ** 2)
    loss, grads = nnx.value_and_grad(loss_fn)(model, rngs)
    optimizer.update(model, grads)
    return loss


@nnx.jit
def eval_step(model, x, y):
    y_pred = model(x, train=False)
    return jnp.mean((y_pred - y) ** 2)


@nnx.jit
def predict_step(model, x):
    return model(x, train=False)


@nnx.scan(in_axes=(nnx.Carry, 0, 0), out_axes=(nnx.Carry, 0))
def train_epoch_scan(carry, xb, yb):
    model, optimizer, rngs = carry
    loss = train_step(model, optimizer, xb, yb, rngs)
    return (model, optimizer, rngs), loss


def split_train_val(X, y, val_frac=0.15, seed=0):
    rng   = np.random.default_rng(seed)
    idx   = rng.permutation(len(X))
    n_val = int(len(X) * val_frac)
    val_idx, train_idx = idx[:n_val], idx[n_val:]
    return X[train_idx], y[train_idx], X[val_idx], y[val_idx]


def train_mlp(model_cls, model_kwargs, X_train, y_train, X_val, y_val,
              epochs=150, batch_size=512, peak_lr=1e-3, final_lr=1e-5,
              weight_decay=0, patience=15, seed=0, verbose=True):
    rngs  = nnx.Rngs(seed)
    model = model_cls(rngs=rngs, **model_kwargs)

    n_train         = X_train.shape[0]
    steps_per_epoch = max(n_train // batch_size, 1)
    total_steps     = steps_per_epoch * epochs

    lr_schedule_fn = optax.warmup_cosine_decay_schedule(
        init_value=0., peak_value=peak_lr,
        warmup_steps=int(total_steps * 0.1),
        decay_steps=int(total_steps * 0.9),
        end_value=final_lr,
    )
    optimizer = nnx.Optimizer(
        model, optax.adamw(learning_rate=lr_schedule_fn, weight_decay=weight_decay), wrt=nnx.Param
    )

    X_train, y_train = jnp.asarray(X_train), jnp.asarray(y_train)
    X_val,   y_val   = jnp.asarray(X_val),   jnp.asarray(y_val)

    shuffle_key = jax.random.key(seed)
    best_val, best_state, bad_epochs = float("inf"), None, 0
    history = {"train_loss": [], "val_loss": []}

    for epoch in tqdm(range(epochs), desc="  epochs", disable=not verbose, leave=False):
        shuffle_key, perm_key = jax.random.split(shuffle_key)
        perm      = jax.random.permutation(perm_key, n_train)
        batch_idx = perm[: steps_per_epoch * batch_size].reshape(steps_per_epoch, batch_size)

        (model, optimizer, rngs), step_losses = train_epoch_scan(
            (model, optimizer, rngs), X_train[batch_idx], y_train[batch_idx]
        )
        train_loss = float(jnp.mean(step_losses))
        val_loss   = float(eval_step(model, X_val, y_val))
        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)

        if val_loss < best_val - 1e-5:
            best_val, bad_epochs = val_loss, 0
            best_state = nnx.state(model)
        else:
            bad_epochs += 1
            if bad_epochs >= patience:
                break

    nnx.update(model, best_state)
    return model, history


def batched_predict(model, X_oh, batch_size=16384):
    chunks = []
    for start in range(0, X_oh.shape[0], batch_size):
        chunks.append(np.asarray(predict_step(model, jnp.asarray(X_oh[start:start + batch_size]))))
    return np.concatenate(chunks)

### 3. Boucle d'entraînement -- 2 CSV × 2 archis = 4 modèles, PLEINE DONNÉE

Pas de cap `N_FIT`/`N_EVAL` cette fois : `fit_idx`/`eval_idx` sont les moitiés ENTIÈRES du split
50/50 (`train_test_split(random_state=0)`), donc jusqu'à ~243k lignes de fit pour `brut` et
~191k pour `trié` (au lieu des 150k plafonnés dans les notebooks précédents). Même split reseedé
identique pour les deux architectures -- comparaison directe, aucune autre variable ne change.

In [ ]:
ARCHS = {
    "shallow": (ShallowProfileMLP, dict(input_dim=L * A)),
    "deep":    (DeepProfileMLP,    dict(L=L, A=A)),
}

results = {}
for tag, df in datasets.items():
    S = seq_matrices[tag]
    y_all = df[SEL_ORG2_COL].to_numpy(np.float64)
    idx_finite = np.flatnonzero(np.isfinite(y_all))

    fit_i, eval_i = train_test_split(idx_finite, test_size=0.5, random_state=0)   # PAS de sous-échantillonnage

    oh = lambda idx: np.eye(A, dtype=np.float32)[S[idx]].reshape(len(idx), -1)
    X_fit,  y_fit  = oh(fit_i),  y_all[fit_i]
    X_eval, y_eval = oh(eval_i), y_all[eval_i]
    Xtr, ytr, Xva, yva = split_train_val(X_fit, y_fit, val_frac=0.15, seed=0)

    for arch_name, (model_cls, kwargs) in ARCHS.items():
        print(f"[{tag:5s} | {arch_name:8s}] train={len(Xtr):,}  val={len(Xva):,}  held-out={len(X_eval):,}", end="  ")
        model, hist = train_mlp(model_cls, kwargs, Xtr, ytr, Xva, yva, seed=0, verbose=False)
        pred_eval = batched_predict(model, X_eval)
        r = pearson(y_eval, pred_eval)
        print(f"-> {len(hist['train_loss'])} epochs, held-out r={r:+.3f}")

        results[(tag, arch_name)] = dict(
            y_eval=y_eval, pred_eval=pred_eval, r=r, history=hist,
            n_fit=len(Xtr), n_eval=len(X_eval),
        )

### 4. Log enrichment prédit vs réel (held-out)

In [ ]:
tags  = list(datasets.keys())
archs = list(ARCHS.keys())

fig, axes = plt.subplots(len(tags), len(archs), figsize=(6 * len(archs), 5.5 * len(tags)))
for row, tag in enumerate(tags):
    for col, arch in enumerate(archs):
        ax = axes[row, col]
        res = results[(tag, arch)]
        y, p = res["y_eval"], res["pred_eval"]
        ax.hexbin(y, p, gridsize=50, bins="log", mincnt=1, cmap="viridis")
        lo, hi = min(y.min(), p.min()), max(y.max(), p.max())
        ax.plot([lo, hi], [lo, hi], "w--", lw=0.9)
        ax.set_title(f"{tag} — {arch}\nr = {res['r']:+.3f}  (n_fit={res['n_fit']:,}, n_eval={res['n_eval']:,})", fontsize=10)
        ax.set_xlabel("log enrichment réel"); ax.set_ylabel("log enrichment prédit")
fig.suptitle("sel_org2 -- prédit vs réel (held-out), Shallow vs Deep, pleine donnée", y=1.01)
fig.tight_layout()
plt.show()

### 5. Recovery top-k%

In [ ]:
fig, axes = plt.subplots(len(tags), len(archs), figsize=(6.2 * len(archs), 5.8 * len(tags)))
for row, tag in enumerate(tags):
    for col, arch in enumerate(archs):
        ax = axes[row, col]
        res = results[(tag, arch)]
        plot_topk_recovery(res["y_eval"], res["pred_eval"], k_frac=0.10,
                            xlabel="log enrichment réel", ylabel="log enrichment prédit",
                            title=f"{tag} — {arch}", ax=ax)
        ax.legend(fontsize=7)
fig.suptitle("sel_org2 -- top-10% recovery (held-out)", y=1.01)
fig.tight_layout()
plt.show()

rows = []
for tag in tags:
    for arch in archs:
        res = results[(tag, arch)]
        row = {"CSV": tag, "archi": arch, "n_fit": res["n_fit"], "n_eval": res["n_eval"],
               "Pearson r": round(res["r"], 3)}
        for frac in (0.01, 0.05, 0.10, 0.20):
            row[f"top-{int(frac*100)}%"] = round(precision_at_k(res["y_eval"], res["pred_eval"], k_frac=frac), 3)
        rows.append(row)
summary = pd.DataFrame(rows).sort_values(["CSV", "archi"]).reset_index(drop=True)
summary

### 6. Deep vs Shallow -- delta

`r`/`top-k%` de `deep` moins `shallow`, par CSV -- positif si l'archi plus profonde exploite
mieux la masse de données disponible.

In [ ]:
pivot = summary.pivot(index="CSV", columns="archi")
metrics = ["Pearson r", "top-1%", "top-5%", "top-10%", "top-20%"]
delta = pd.DataFrame({m: pivot[(m, "deep")] - pivot[(m, "shallow")] for m in metrics})
delta.columns = [f"Δ {m} (deep − shallow)" for m in metrics]
delta

### 7. Rappel -- comparaison au ProfileMLP standard capé à 150k

`AAV5_SEL_profile_model_before_after_sorting.ipynb` avait donné, avec `ShallowProfileMLP` capé
à `N_FIT=N_EVAL=150 000` : `sel_org2` brut r=+0.407, trié r=+0.438. Comparer ces deux chiffres à
la colonne `shallow` de la table ci-dessus isole l'effet du volume de fit seul (même archi,
plus de données) ; la colonne `deep` isole en plus l'effet de l'architecture.